# DSC 106 Project 3 — CMIP6 checkpoint data pipeline
**Question:** When will your part of the world cross 2°C?

**How to use:** fill in the team details in the next code cell, then choose **Runtime → Run all**. The first run takes roughly 15–40 minutes, depending on Colab's connection.

The last cell downloads `dsc106_p3_outputs.zip`, which holds six exploratory figures, the data for the D3 page, and a voiceover script filled in with the numbers computed here.

**Data:** monthly near-surface air temperature (`tas`, table `Amon`) from the public CMIP6 archive at `gs://cmip6`.

In [ ]:
%pip install -q xarray zarr gcsfs dask cartopy

In [ ]:
# ===== EDIT THESE =====
TEAM_NAME = "Your Team Name"
TEAM_MEMBERS = ["Evelyn Na", "Teammate 2", "Teammate 3", "Teammate 4"]
# ======================

VIS_TITLE = "When Will Your Part of the World Cross 2°C?"
N_MODELS = 8                  # lower to 5 if the run is too slow
PRIMARY = "ssp245"            # scenario used for the single-scenario figures
BASELINE = (1850, 1900)       # pre-industrial reference period
END_WINDOW = (2081, 2100)     # end-of-century window, as used by the IPCC
ARCTIC_LAT = 66.5             # Arctic Circle
TROPIC_LAT = 23.5
THRESHOLDS = [1.5, 2.0, 3.0]  # warming levels in °C
SEARCH_FROM = 1970            # earliest crossing year counted
PLACES = {"San Diego": (32.7, -117.2), "Seoul": (37.6, 127.0), "Chicago": (41.9, -87.6), "Mumbai": (19.1, 72.9),
          "Paris": (48.9, 2.4), "Lagos": (6.5, 3.4), "São Paulo": (-23.5, -46.6), "Sydney": (-33.9, 151.2)}

SCENARIOS = {
    "ssp126": {"label": "SSP1-2.6", "desc": "Low emissions", "color": "#2a78d6"},
    "ssp245": {"label": "SSP2-4.5", "desc": "Intermediate", "color": "#1a9e6f"},
    "ssp370": {"label": "SSP3-7.0", "desc": "High", "color": "#e0a100"},
    "ssp585": {"label": "SSP5-8.5", "desc": "Very high", "color": "#c8372d"},
}
PREFERRED_MODELS = ["MPI-ESM1-2-LR", "MIROC6", "MRI-ESM2-0", "ACCESS-ESM1-5", "NorESM2-LM",
                    "IPSL-CM6A-LR", "CanESM5", "CESM2-WACCM", "GFDL-ESM4", "INM-CM5-0",
                    "EC-Earth3", "FGOALS-g3", "BCC-CSM2-MR", "CMCC-ESM2", "TaiESM1"]

In [ ]:
import json, os, re, shutil, time, zipfile
import gcsfs
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

OUT = Path("outputs"); FIGS = OUT / "figs"; DATA = OUT / "data"; CACHE = Path("cache")
for p in (FIGS, DATA, CACHE):
    p.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = ["historical"] + list(SCENARIOS)
LAT = np.arange(-88.75, 90, 2.5)    # common 2.5 degree grid (cell centres)
LON = np.arange(-178.75, 180, 2.5)
WEIGHTS = np.cos(np.deg2rad(LAT))   # grid cells shrink towards the poles
YEARS = np.arange(1850, 2101)

plt.rcParams.update({"figure.dpi": 150, "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e4df",
                     "grid.linewidth": 0.7, "axes.axisbelow": True, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "figure.facecolor": "white"})

try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    HAS_CARTOPY = True
    try:
        list(cfeature.NaturalEarthFeature("physical", "coastline", "110m").geometries())[:1]
        HAS_COAST = True
    except Exception:
        HAS_COAST = False
except Exception:
    HAS_CARTOPY = HAS_COAST = False
print("cartopy:", HAS_CARTOPY, "| coastlines:", HAS_COAST)

## 1. Choose the model runs
Every model must have the historical run and all four future scenarios, from the same ensemble member, so that scenarios are compared fairly.

In [ ]:
def load_catalog():
    urls = ["https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv",
            "https://storage.googleapis.com/cmip6/pangeo-cmip6.csv"]
    last = None
    for url in urls:
        try:
            cat = pd.read_csv(url)
            print("catalog:", url, f"({len(cat):,} rows)")
            return cat
        except Exception as e:
            last = e
    raise RuntimeError(f"Could not read the CMIP6 catalog: {last}")

def member_key(member):
    nums = [int(n) for n in re.findall(r"\d+", member)]
    return (member != "r1i1p1f1", nums)

def pick_runs(cat):
    q = cat[(cat.table_id == "Amon") & (cat.variable_id == "tas") & cat.experiment_id.isin(EXPERIMENTS)]
    complete = {}
    for (model, member), g in q.groupby(["source_id", "member_id"]):
        if set(EXPERIMENTS) <= set(g.experiment_id):
            complete.setdefault(model, []).append(member)
    rank = {m: i for i, m in enumerate(PREFERRED_MODELS)}
    models = sorted(complete, key=lambda m: (rank.get(m, 999), m))
    runs = {}
    for model in models:
        member = sorted(complete[model], key=member_key)[0]
        g = q[(q.source_id == model) & (q.member_id == member)]
        stores = {}
        for exp in EXPERIMENTS:
            e = g[g.experiment_id == exp].copy()
            e["_gn"] = e.grid_label != "gn"       # prefer the native grid
            sort_cols = ["_gn"] + (["version"] if "version" in e else [])
            asc = [True] + ([False] if "version" in e else [])
            stores[exp] = e.sort_values(sort_cols, ascending=asc).iloc[0].zstore
        runs[model] = {"member": member, "stores": stores}
    print(len(runs), "models have all five experiments")
    return runs

catalog = load_catalog()
candidates = pick_runs(catalog)
print(list(candidates)[:N_MODELS + 4])

## 2. Load and regrid
Each run becomes annual means on a common 2.5° grid. Models use different grids, so this step is needed before they can be averaged. Results are cached, so re-running after a disconnect resumes where it stopped.

In [ ]:
def to_common_grid(da):
    renames = {k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in da.dims}
    da = da.rename(renames).sortby("lat")
    da = da.assign_coords(lon=da.lon % 360).sortby("lon")
    da = da.isel(lon=~pd.Index(da.lon.values).duplicated())
    # Wrap one column around the date line so interpolation works there
    left = da.isel(lon=-1).assign_coords(lon=float(da.lon[-1]) - 360)
    right = da.isel(lon=0).assign_coords(lon=float(da.lon[0]) + 360)
    wrapped = xr.concat([left, da, right], dim="lon").transpose("year", "lat", "lon")
    target_lat = np.clip(LAT, float(da.lat.min()), float(da.lat.max()))
    out = wrapped.interp(lat=target_lat, lon=LON % 360, method="linear")
    return out.values.astype("float32")

GCS = gcsfs.GCSFileSystem(token="anon")

def load_annual(zstore):
    ds = xr.open_zarr(GCS.get_mapper(zstore), consolidated=True)
    da = ds["tas"].squeeze(drop=True)
    da = da.drop_vars([c for c in ("height",) if c in da.coords])
    months = da["time"].groupby("time.year").count()
    annual = da.groupby("time.year").mean("time").load()
    full_years = months.year.values[months.values == 12]
    annual = annual.sel(year=[y for y in annual.year.values if y in set(full_years) and y <= 2100])
    return annual.year.values, to_common_grid(annual) - 273.15

def get_run(model, exp, zstore):
    path = CACHE / f"{model}__{exp}.npz"
    if path.exists():
        z = np.load(path)
        return z["years"], z["tas"]
    years, tas = load_annual(zstore)
    np.savez(path, years=years, tas=tas)
    return years, tas

fields, used = {}, {}
for model, info in candidates.items():
    if len(fields) >= N_MODELS:
        break
    t0 = time.time()
    try:
        runs = {exp: get_run(model, exp, info["stores"][exp]) for exp in EXPERIMENTS}
    except Exception as e:
        print(f"skip {model}: {type(e).__name__}: {str(e)[:100]}")
        continue
    hy, ht = runs["historical"]
    base = ht[(hy >= BASELINE[0]) & (hy <= BASELINE[1])].mean(0)   # each model's own baseline
    fields[model] = {}
    for scen in SCENARIOS:
        grid = np.full((len(YEARS), len(LAT), len(LON)), np.nan, dtype="float32")
        for yrs, tas in (runs["historical"], runs[scen]):
            keep = (yrs >= YEARS[0]) & (yrs <= YEARS[-1])
            grid[yrs[keep] - YEARS[0]] = tas[keep] - base
        fields[model][scen] = grid
    used[model] = info["member"]
    print(f"{model:18s} {info['member']:10s} {time.time() - t0:5.0f}s  ({len(fields)}/{N_MODELS})")
assert len(fields) >= 3, "Too few models loaded — check the messages above."
MODELS = list(fields)

## 3. Compute warming and crossing years
Warming is the change from 1850–1900, using each model's own baseline. A place "crosses" a threshold in the first year from 1970 on that its 20-year average reaches that level, and the year reported is the middle of that 20-year window. Earlier local swings are ignored as natural variability. Using 20 years removes year-to-year noise, so the latest year we can report is about 2090. The map shows the median across models.

In [ ]:
ens = {s: np.stack([fields[m][s] for m in MODELS]) for s in SCENARIOS}   # model, year, lat, lon
mmm = {s: np.nanmean(ens[s], axis=0) for s in SCENARIOS}                 # multi-model mean

REGIONS = {"Global": LAT > -90, "Arctic": LAT >= ARCTIC_LAT,
           "Tropics": np.abs(LAT) <= TROPIC_LAT, "Antarctic": LAT <= -ARCTIC_LAT}

def area_mean(arr, mask):
    """Area-weighted mean over the last two axes (lat, lon) for the latitudes in mask."""
    w = WEIGHTS[mask]
    return (np.nanmean(arr[..., mask, :], axis=-1) * w).sum(-1) / w.sum()

def first_crossing(arr, threshold, year_axis=0, window=20):
    """First centre year of a 20-year average at or above threshold; NaN if never."""
    arr = np.moveaxis(arr, year_axis, 0)
    good = np.isfinite(arr)
    zero = np.zeros((1,) + arr.shape[1:])
    total = np.concatenate([zero, np.cumsum(np.where(good, arr, 0.0), axis=0)])
    count = np.concatenate([zero, np.cumsum(good, axis=0)])
    n = count[window:] - count[:-window]
    with np.errstate(invalid="ignore", divide="ignore"):
        avg = np.where(n >= window - 5, (total[window:] - total[:-window]) / n, np.nan)
    centres = YEARS[:len(avg)] + window // 2
    # Ignore early windows: a 2°C local swing before 1970 is natural variability, not the warming trend
    above = (avg >= threshold) & (centres.reshape((-1,) + (1,) * (avg.ndim - 1)) >= SEARCH_FROM)
    return np.where(above.any(0), centres[above.argmax(0)].astype(float), np.nan)

def summarize_crossing(years):
    """years has models on axis 0. Never-crossing models count as later than any year."""
    filled = np.where(np.isnan(years), np.inf, years)
    median = np.median(filled, axis=0)
    earliest = filled.min(0)
    latest = filled.max(0)
    return median, earliest, latest, np.isfinite(filled).sum(0)

def cell_of(lat, lon):
    return int(np.abs(LAT - lat).argmin()), int(np.abs(LON - lon).argmin())

def clean_year(v):
    return int(v) if np.isfinite(v) else None

series = {s: {r: area_mean(ens[s], mask) for r, mask in REGIONS.items()} for s in SCENARIOS}
end_mask = (YEARS >= END_WINDOW[0]) & (YEARS <= END_WINDOW[1])
end_mean = {s: np.nanmean(mmm[s][end_mask], axis=0) for s in SCENARIOS}

crossing, cross_global = {}, {}
for s in SCENARIOS:
    crossing[s], cross_global[s] = {}, {}
    for t in THRESHOLDS:
        by_model = np.stack([first_crossing(ens[s][i], t) for i in range(len(MODELS))])
        crossing[s][t] = summarize_crossing(by_model)
        g = np.array([first_crossing(series[s]["Global"][i], t) for i in range(len(MODELS))])
        cross_global[s][t] = [clean_year(float(v)) for v in summarize_crossing(g)[:3]]

facts = {"models": MODELS, "members": used, "n_models": len(MODELS), "global": {}, "places": {}, "world_2c": {}}
for s in SCENARIOS:
    facts["global"][s] = {f"{t:g}": cross_global[s][t] for t in THRESHOLDS}
    med = crossing[s][2.0][0]
    area = np.outer(WEIGHTS, np.ones(len(LON))); area = area / area.sum()
    facts["world_2c"][s] = {
        "share_by_2050": round(100 * float(area[med <= 2050].sum())),
        "share_never": round(100 * float(area[~np.isfinite(med)].sum())),
    }
for name, (la, lo) in PLACES.items():
    i, j = cell_of(la, lo)
    facts["places"][name] = {s: [clean_year(float(a[i, j])) for a in crossing[s][2.0][:3]] for s in SCENARIOS}
mp = crossing[PRIMARY][2.0]
spread_years = np.where(np.isfinite(mp[1]), np.minimum(mp[2], 2100) - mp[1], np.nan)
print(json.dumps({"global": facts["global"], "world_2c": facts["world_2c"], "places": facts["places"]}, indent=1))

## 4. Six exploratory graphs
Each graph answers a smaller question that leads to the final question.

In [ ]:
P = SCENARIOS[PRIMARY]
SUB = f"{len(MODELS)}-model CMIP6 mean, change from {BASELINE[0]}–{BASELINE[1]}"
WARM = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "warming", [(0, "#2a78d6"), (3 / 17, "#ebe9e4"), (6 / 17, "#f2b35e"), (11 / 17, "#c8372d"), (1, "#4f0d16")])
VMIN, VMAX = -3, 14
YEAR_MAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "years", ["#4f0d16", "#a3201f", "#d9592f", "#f2a65a", "#fbe3b5"])
YEAR_MAP.set_bad("#bdbbb4")
Y0, Y1 = 2000, 2090

def save(fig, name):
    fig.savefig(FIGS / name, bbox_inches="tight", facecolor="white")
    plt.show()

def map_axes(fig, pos):
    return fig.add_subplot(pos, projection=ccrs.EqualEarth()) if HAS_CARTOPY else fig.add_subplot(pos)

def draw_map(ax, values, **kw):
    if HAS_CARTOPY:
        m = ax.pcolormesh(LON, LAT, values, transform=ccrs.PlateCarree(), shading="nearest", rasterized=True, **kw)
        if HAS_COAST:
            ax.coastlines(linewidth=0.5, color="#222")
        ax.set_global()
    else:
        m = ax.pcolormesh(LON, LAT, values, shading="nearest", rasterized=True, **kw)
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(False)
    return m

# Graph 1
fig, ax = plt.subplots(figsize=(10, 5.4))
for t in (1.5, 2.0):
    ax.axhline(t, color="#777", lw=0.9, ls="--")
    ax.annotate(f"{t:g}°C", (1852, t), xytext=(0, 3), textcoords="offset points", fontsize=9, color="#555")
past = YEARS <= 2014
arr = series[PRIMARY]["Global"]
ax.fill_between(YEARS[past], np.nanmin(arr[:, past], 0), np.nanmax(arr[:, past], 0), color="#777", alpha=0.2, lw=0)
ax.plot(YEARS[past], np.nanmean(arr[:, past], 0), color="#222", lw=2)
ax.annotate("Historical", (1950, 0.75), fontsize=10, color="#222")
for s, meta in SCENARIOS.items():
    arr = series[s]["Global"][:, ~past]
    ax.fill_between(YEARS[~past], np.nanmin(arr, 0), np.nanmax(arr, 0), color=meta["color"], alpha=0.14, lw=0)
    mean = np.nanmean(arr, 0)
    ax.plot(YEARS[~past], mean, color=meta["color"], lw=2)
    last = mean[np.isfinite(mean)][-1]
    ax.annotate(meta["label"], (2100, last), xytext=(6, 0), textcoords="offset points", va="center", fontsize=10)
ax.set(xlim=(1850, 2100), ylabel="Warming (°C)")
ax.set_title(f"1. How much does the world warm under each scenario?\n{SUB}; bands show the model range", fontsize=12)
save(fig, "fig1_global_by_scenario.png")

# Graph 2
fig = plt.figure(figsize=(10, 5.4)); ax = map_axes(fig, 111)
m = draw_map(ax, end_mean[PRIMARY], cmap=WARM, vmin=VMIN, vmax=VMAX)
fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.05, pad=0.05, label="Warming (°C)")
ax.set_title(f"2. Which regions warm the most by 2100?\n{P['label']}, {END_WINDOW[0]}–{END_WINDOW[1]}; {SUB}", fontsize=12)
save(fig, "fig2_map_end_century.png")

# Graph 3 (sine axis: equal height means equal Earth area)
fig, ax = plt.subplots(figsize=(10, 5.4))
for s, meta in SCENARIOS.items():
    ax.plot(np.nanmean(end_mean[s], axis=1), np.sin(np.deg2rad(LAT)), color=meta["color"], lw=2,
            label=f"{meta['label']} ({meta['desc'].lower()})")
ticks = [-90, -60, -30, 0, 30, 60, 90]
ax.set_yticks(np.sin(np.deg2rad(ticks)), [f"{abs(t)}°{'N' if t > 0 else 'S' if t < 0 else ''}" for t in ticks])
ax.set(xlabel="Warming (°C)", ylabel="Latitude (spaced by Earth's area)")
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"3. How does warming change by latitude?\nAverage across each latitude, {END_WINDOW[0]}–{END_WINDOW[1]}; {SUB}", fontsize=12)
save(fig, "fig3_warming_by_latitude.png")

# Graph 4
fig, ax = plt.subplots(figsize=(10, 5.4))
rows = [(s, t) for s in SCENARIOS for t in (1.5, 2.0)]
for row, (s, t) in enumerate(rows):
    med, lo, hi = cross_global[s][t]
    color = SCENARIOS[s]["color"]
    if lo is not None:
        ax.plot([lo, hi if hi else 2090], [row, row], color=color, lw=2.5, alpha=0.4)
        if hi is None:
            ax.scatter([2090], [row], color=color, marker=">", s=45, alpha=0.6)
    if med is None:
        ax.annotate("Median model doesn't reach it by about 2090", (2092, row), xytext=(0, 6),
                    textcoords="offset points", ha="right", fontsize=9, color="#444")
    if med is not None:
        ax.scatter([med], [row], color=color, s=70, zorder=3, marker="o" if t == 1.5 else "D", edgecolor="white")
ax.set_yticks(range(len(rows)), [f"{SCENARIOS[s]['label']}, {t:g}°C" for s, t in rows])
ax.invert_yaxis(); ax.set_xlim(1990, 2095); ax.set_xlabel("Crossing year (middle of the first 20-year average at that level)")
ax.set_title(f"4. When does each scenario cross 1.5°C and 2°C globally?\nMarkers are the model median; lines show the earliest to latest model", fontsize=12)
save(fig, "fig4_global_crossing.png")

# Graph 5
fig = plt.figure(figsize=(10, 5.4)); ax = map_axes(fig, 111)
vals = np.ma.masked_invalid(np.where(np.isfinite(mp[0]), mp[0], np.nan))
m = draw_map(ax, vals, cmap=YEAR_MAP, vmin=Y0, vmax=Y1)
fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.05, pad=0.05, extend="min", label="Year the local 20-year average reaches 2°C")
ax.set_title(f"5. Do different regions cross 2°C at different times?\n{P['label']}, median of {len(MODELS)} models; gray = not reached by about 2090", fontsize=12)
save(fig, "fig5_crossing_map.png")

# Graph 6
fig = plt.figure(figsize=(10, 5.4)); ax = map_axes(fig, 111)
PURPLE = plt.get_cmap("Purples").copy(); PURPLE.set_bad("#bdbbb4")
m = draw_map(ax, np.ma.masked_invalid(spread_years), cmap=PURPLE, vmin=0, vmax=100)
fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.05, pad=0.05, extend="max", label="Years between the earliest and latest model")
ax.set_title(f"6. How much do the models disagree on timing?\n{P['label']}, 2°C; models that never cross count as 2100; gray = no model crosses", fontsize=12)
save(fig, "fig6_timing_disagreement.png")

## 5. Export for D3
Crossing years plus decade means, in one static JSON file, so no server code is needed. A 0 means "not reached by about 2090".

In [ ]:
DECADES = list(range(1850, 2100, 10))

def decade_grid(a):
    frames = [np.nanmean(a[(YEARS >= d) & (YEARS < d + 10)], axis=0) for d in DECADES]
    return np.rint(np.stack(frames) * 10).astype(int)

def decade_line(a):
    return [round(float(np.nanmean(a[(YEARS >= d) & (YEARS < d + 10)])), 2) for d in DECADES]

def year_grid(a):
    return np.where(np.isfinite(a), a, 0).astype(int).ravel().tolist()

payload = {
    "lat": LAT.tolist(), "lon": LON.tolist(), "decades": DECADES, "models": MODELS, "n_models": len(MODELS),
    "baseline": list(BASELINE), "cell": 2.5, "scale": 0.1, "thresholds": THRESHOLDS,
    "scenarios": [{"id": s, "label": m["label"], "desc": m["desc"], "color": m["color"]} for s, m in SCENARIOS.items()],
    "places": [{"name": n, "lat": la, "lon": lo} for n, (la, lo) in PLACES.items()],
    "grid": {s: decade_grid(mmm[s]).ravel().tolist() for s in SCENARIOS},
    "global": {s: decade_line(np.nanmean(series[s]["Global"], axis=0)) for s in SCENARIOS},
    "global_crossing": {s: {f"{t:g}": cross_global[s][t] for t in THRESHOLDS} for s in SCENARIOS},
    "crossing": {s: {f"{t:g}": {"median": year_grid(crossing[s][t][0]), "earliest": year_grid(crossing[s][t][1]),
                              "latest": year_grid(crossing[s][t][2]), "count": crossing[s][t][3].ravel().tolist()}
                     for t in THRESHOLDS} for s in SCENARIOS},
}
with open(DATA / "warming.json", "w") as f:
    json.dump(payload, f, separators=(",", ":"))
with open(DATA / "facts.json", "w") as f:
    json.dump(facts, f, indent=1)
meta = {"team": TEAM_NAME, "members": TEAM_MEMBERS, "title": VIS_TITLE, "primary": PRIMARY,
        "end_window": list(END_WINDOW), "baseline": list(BASELINE)}
with open(DATA / "meta.json", "w") as f:
    json.dump(meta, f, indent=1)
print(f"warming.json: {(DATA / 'warming.json').stat().st_size / 1e6:.1f} MB")

## 6. Voiceover script
Every number comes from the results above. Read it against the graphs, then rewrite it in your own words before recording.

In [ ]:
def when(y):
    return f"around {y}" if y else "not until after about 2090"
def reaches(name, y):
    return f"{name} reaches 2 degrees around {y}" if y else f"{name} doesn't reach 2 degrees by about 2090"
g15, g20 = cross_global[PRIMARY][1.5], cross_global[PRIMARY][2.0]
low20, high20 = cross_global["ssp126"][2.0], cross_global["ssp585"][2.0]
sd, seoul = facts["places"]["San Diego"][PRIMARY], facts["places"]["Seoul"][PRIMARY]
w = facts["world_2c"][PRIMARY]
members = ", ".join(TEAM_MEMBERS[:-1]) + (", and " if len(TEAM_MEMBERS) > 1 else "") + TEAM_MEMBERS[-1]
lines = [
 ("Title slide", f"We're {TEAM_NAME}: {members}."),
 ("Dataset", f"We chose CMIP6, the climate model archive hosted on Google Cloud. We use monthly near-surface air temperature from {len(MODELS)} models, from historical runs and four future emissions scenarios. We average to yearly values on a shared two-and-a-half degree grid, and measure warming from {BASELINE[0]} to {BASELINE[1]}."),
 ("Graph 1", "First, how much does the world warm under each scenario? The paths stay close at first, then separate."),
 ("Graph 2", f"Next, which regions warm most by 2100? This {P['label']} map shows warming isn't even across the planet."),
 ("Graph 3", "Averaging by latitude shows how that pattern changes from pole to pole."),
 ("Graph 4", f"Then we asked when. Under {P['label']}, the world crosses 1.5 degrees {when(g15[0])} and 2 degrees {when(g20[0])}. {reaches('The median model in the low scenario', low20[0])}, and the highest scenario gets there {when(high20[0])}."),
 ("Graph 5", f"Different places cross at different times. Under {P['label']}, about {w['share_by_2050']} percent of Earth's area reaches 2 degrees by 2050, while about {w['share_never']} percent still hasn't by about 2090."),
 ("Graph 6", "Models also disagree on timing, so we want to show that range, not just one year."),
 ("Question", f"That leads to our title and question: {VIS_TITLE}"),
 ("Prototype", f"Here's our D3 prototype. You choose a threshold and a scenario. Darker places cross earlier. Under {P['label']}, {reaches('San Diego', sd[0])}, and {reaches('Seoul', seoul[0])}. Clicking a place shows its warming and the range across models. Next, we'll refine the design and add annotations."),
]
script = "# Voiceover script\n\n" + "\n\n".join(f"**[{k}]** {v}" for k, v in lines) + "\n"
(OUT / "voiceover_script.md").write_text(script)
words = len(" ".join(v for _, v in lines).split())
print(script); print(f"{words} words, about {words / 2.4:.0f} seconds at a steady pace")

In [ ]:
archive = "dsc106_p3_outputs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("Saved", archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass